## Serialize a Vector Data using SchemaOrg and the CUAHSI.org-Raster extension

The purpose of this notebook is to evaluate how Vector data can be extracted and mapped to our Pydantic classes.

In [18]:
import os
import sys
import fiona
import hashlib
import geopandas
import mimetypes
from glob import glob
from pathlib import Path

# add the parent directory to the path. This is the 
# directory that contains our pydantic classes.
sys.path.append('..')
import base
import core
import dataset
import vector

In [19]:
def compute_sha256(file_path: Path) -> str:
    """Computes the SHA256 hash of a file.

    Args:
        file_path: The path to the file.

    Returns:
        The hexadecimal representation of the SHA256 hash.
    """
    sha256_hash = hashlib.sha256()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            sha256_hash.update(chunk)
    return sha256_hash.hexdigest()

In [20]:
# Add Shapefile MIME type if not already present
mimetypes.add_type("application/x-esri-shapefile", ".shp")
mimetypes.add_type("application/x-esri-shx", ".shx")
mimetypes.add_type("application/x-dbase", ".dbf")
mimetypes.add_type("text/plain", ".prj")
mimetypes.add_type("text/plain", ".cpg")
mimetypes.add_type("text/plain", ".asc")
mimetypes.add_type("application/geo+json", ".geojson")
mimetypes.add_type("application/gml+xml", ".gml")


In [27]:

def encode_vector_metadata(filepath, validate_bbox=True):

    # get all file names that match the patter of the input filepath
    search_path = f"{'.'.join(filepath.split('.')[:-1])}.*"
    associated_files = glob(search_path)

    # Read the Shapefile
    gdf = geopandas.read_file(filepath)

    feature_count = len(gdf)

    fields = {}
    field_names = list(gdf.columns)
    for fname in field_names:
        meta = {}
        meta.update({'dtype': str(gdf[fname].dtype)})
    
        # try to get min and max values with exception 
        # handling because some fields may not support
        # reduce, e.g. geometry.
        try:
            meta.update({'min_value': gdf[fname].min().item()})
            meta.update({'max_value': gdf[fname].max().item()})
        except:
            pass
        fields[fname] = meta
        
    geometry_type = gdf.geom_type.unique()[0]
    
    # extent
    extent_west, extent_south, extent_east, extent_north = gdf.total_bounds
    
    box_str = f'{extent_south} {extent_west} {extent_north} {extent_east}'
    geo = base.GeoShape(box = box_str, validate_bbox=validate_bbox)

    # srs
    srs = None
    if gdf.crs is not None:
        # crs_datum = gdf.crs.datum.name
        # crs_unit = gdf.crs.axis_info[0].unit_name
        crs_type = "Geographic" if gdf.crs.is_geographic else "Projected"

        srs = base.SpatialReference(
            name = gdf.crs.name,
            srsType=crs_type,
            code=gdf.crs.srs,
            wktString=gdf.crs.to_wkt()
        )
    
    place = base.Place(
        geo=geo,
        srs=srs,
    )

    variables = []
    for field_name, field_values in fields.items():
        
        minValue = field_values['min_value'] if 'min_value' in field_values else None
        maxValue = field_values['max_value'] if 'max_value' in field_values else None
            
        variable = base.PropertyValue(
            name = field_name,
            propertyID = field_name,
            value = field_values['dtype'],
            minValue = minValue,
            maxValue = maxValue)
        variables.append(variable)


    files = []
    for fpath in associated_files:
        print(fpath)
        files.append(
            base.MediaObject(
                contentUrl = f'https://hydroshare.org/my-resource/{fpath}',
                name = Path(fpath).name,
                sha256 = compute_sha256(Path(fpath)),
                contentSize = f'{os.path.getsize(Path(fpath))/1024} KB',
                encodingFormat = mimetypes.guess_type(Path(fpath))[0],
            )
        )
    
    return vector.GeographicVector(
        featureCount = feature_count,
        fieldCount = len(fields.keys()),
        geometryType = geometry_type,
        variableMeasured = variables,
        associatedMedia=files,
        spatialCoverage=place,
    )


### Encode a Shapefile

In [26]:
meta = encode_vector_metadata('data/watershed.shp', validate_bbox=True)
print(meta.model_dump_json(exclude_none=True, indent=4))

data/watershed.prj
data/watershed.xsd
data/watershed.shp
data/watershed.qmd
data/watershed.shx
data/watershed.dbf
{
    "context": "https://hydroshare.org/schema",
    "type": "GeographicVector",
    "spatialCoverage": {
        "type": "Place",
        "geo": {
            "type": "GeoShape",
            "box": "39.84727077900004 -105.700627563 40.15893526200006 -104.933629104"
        },
        "srs": {
            "type": "SpatialReference",
            "name": "WGS 84",
            "srsType": "geographic",
            "code": "EPSG:4326",
            "wktString": "GEOGCRS[\"WGS 84\",ENSEMBLE[\"World Geodetic System 1984 ensemble\",MEMBER[\"World Geodetic System 1984 (Transit)\"],MEMBER[\"World Geodetic System 1984 (G730)\"],MEMBER[\"World Geodetic System 1984 (G873)\"],MEMBER[\"World Geodetic System 1984 (G1150)\"],MEMBER[\"World Geodetic System 1984 (G1674)\"],MEMBER[\"World Geodetic System 1984 (G1762)\"],MEMBER[\"World Geodetic System 1984 (G2139)\"],MEMBER[\"World Geodetic Sys

/Users/castro/Documents/work/ciroh/HydroShare-Cloud-Native-Data/schema/.venv/lib/python3.10/site-packages/pyproj/crs/crs.py:1295: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj = self._crs.to_proj4(version=version)


### Encode GeoJSON

In [24]:
meta = encode_vector_metadata('data/watershed_json.geojson', validate_bbox=True)
print(meta.model_dump_json(exclude_none=True, indent=4))

AttributeError: 'CRS' object has no attribute 'to_proj'

### Encode GML

In [7]:
meta = encode_vector_metadata('data/watershed_gml.gml', validate_bbox=True)
print(meta.model_dump_json(exclude_none=True, indent=4))

data/watershed_gml.gfs
data/watershed_gml.gml
{
    "context": "https://hydroshare.org/schema",
    "type": "GeographicVector",
    "spatialCoverage": {
        "type": "Place",
        "geo": {
            "type": "GeoShape",
            "box": "39.847270779 -105.700627563 40.1589352620001 -104.933629104"
        },
        "additionalProperty": [
            {
                "type": "PropertyValue",
                "name": "Geographic Coordinate System",
                "value": "WGS 84 (EPSG:4326)"
            },
            {
                "type": "PropertyValue",
                "name": "Datum",
                "value": "World Geodetic System 1984 ensemble"
            },
            {
                "type": "PropertyValue",
                "name": "Unit",
                "value": "degree"
            },
            {
                "type": "PropertyValue",
                "name": "Coordinate String",
                "value": "GEOGCRS[\"WGS 84\",ENSEMBLE[\"World Geodetic Syst